# Mini-Batch DRGU Convergence Verification

This notebook verifies that the mini-batch Fisher scoring implementation produces nearly identical parameter estimates to the original full-batch PyTorch DRGU implementation.

## Objectives
1. Compare parameter estimates between full-batch and mini-batch DRGU
2. Verify convergence under different scenarios
3. Analyze convergence diagnostics and optimization behavior
4. Test robustness across sample sizes and data characteristics


In [1]:
import torch
import numpy as np
import pandas as pd

# Import DRGU implementations
from robustinfer.drgu import DRGU as DRGUTorch
from robustinfer import DRGUMiniBatch

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✓ Imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


✓ Imports successful
PyTorch version: 2.8.0+cpu
Device: cpu


## Data Generation

Generate synthetic data with known treatment effects for DRGU estimation.


In [2]:
def sample_data():
    """Create sample data for testing"""
    np.random.seed(42)
    n = 100

    # Generate covariates
    x1 = np.random.normal(0, 1, n)
    x2 = np.random.normal(0, 1, n)

    # Generate treatment (binary)
    treatment_prob = 1 / (1 + np.exp(-(0.5 + 0.3 * x1 + 0.2 * x2)))
    z = np.random.binomial(1, treatment_prob, n)

    # Generate response
    y = 0.5 + 0.0 * z + 0.3 * x1 + 0.2 * x2 + np.random.normal(0, 0.5, n)

    # Create DataFrame
    data = pd.DataFrame({'x1': x1, 'x2': x2, 'treatment': z, 'response': y})

    return data


In [3]:
df = sample_data()

In [4]:
df.head()

,x1,x2,treatment,response
0,0.496714,-1.415371,0,0.290048
1,-0.138264,-0.420645,0,0.668550
2,0.647689,-0.342715,0,0.766260
3,1.523030,-0.802277,0,0.485104
4,-0.234153,-0.161286,1,0.293436


## Fit original DRGU


In [5]:
print("Fitting PyTorch model (CPU)...")
model_torch_cpu = DRGUTorch(df, covariates=["x1", "x2"], treatment="treatment", response="response", device='cpu')
model_torch_cpu.fit(tol=1e-5, lamb = 1e-6)

Fitting PyTorch model (CPU)...
Step 0 gradient norm: 0.195176
Step 10 gradient norm: 0.006927
Did not converge, norm step = 0.000465608318336308


In [6]:
model_torch_cpu.summary()

,Names,Coefficient,Null_Hypothesis,Std_Error,Z_Score,P_Value
0,delta,0.485595,0.5,0.046693,-0.308496,0.757705
1,beta_0,0.607535,0.0,0.215253,2.822430,0.004766
2,beta_1,0.468635,0.0,0.234681,1.996905,0.045836
3,beta_2,0.237299,0.0,0.218240,1.087332,0.276890
4,gamma_0,-0.006471,0.0,0.320474,-0.020192,0.983890
5,gamma_1,0.953004,0.0,0.578352,1.647793,0.099395
6,gamma_2,0.879849,0.0,0.548627,1.603729,0.108774
7,gamma_3,-0.685641,0.0,0.724444,-0.946437,0.343925
8,gamma_4,-0.818478,0.0,0.602619,-1.358202,0.174400


### Fit minibatch 

In [7]:
model_minibatch = DRGUMiniBatch(df, covariates=["x1", "x2"], treatment="treatment", response="response")

## fit with full pairs

In [9]:
model_minibatch.fit(tol = 1e-3, pairs_per_anchor=99, pairs_per_batch = 5000, option='plain', lamb = 0.0, max_epochs = 20)

{'delta': tensor([0.4855]),
 'beta': tensor([0.6075, 0.4683, 0.2367]),
 'gamma': tensor([-0.0089,  0.9660,  0.9025, -0.6981, -0.8439]),
 'converged': True,
 'iterations': 5,
 'final_delta_norm': 0.0007753105213244756,
 'epochs_run': 5}

In [10]:
model_minibatch.estimate_variance(pairs_per_anchor = 99, s=100)

Variance estimation: s=100, k=99
MC estimation: n=100, s=100, m=99
  Batch 20
  Batch 40
  Batch 60
  Batch 80
  Batch 100
Anchor-based variance computed
Variance matrix computed


In [11]:
model_minibatch.summary()

,Names,Coefficient,Null_Hypothesis,Std_Error,Z_Score,P_Value
0,delta,0.485517,0.5,0.110528,-0.131035,0.895747
1,beta_0,0.607463,0.0,0.217415,2.794025,0.005206
2,beta_1,0.468259,0.0,0.236973,1.976004,0.048154
3,beta_2,0.236750,0.0,0.220231,1.075007,0.282372
4,gamma_0,-0.008887,0.0,0.325323,-0.027316,0.978207
5,gamma_1,0.965956,0.0,0.594495,1.624834,0.104198
6,gamma_2,0.902479,0.0,0.571784,1.578355,0.114484
7,gamma_3,-0.698112,0.0,0.741624,-0.941329,0.346536
8,gamma_4,-0.843858,0.0,0.628325,-1.343027,0.179263


## Large batch
momentum works better

## Fit with momentum

In [12]:
model_minibatch = DRGUMiniBatch(df, covariates=["x1", "x2"], treatment="treatment", response="response")

In [14]:
model_minibatch.fit(pairs_per_anchor=10, pairs_per_batch = 5000, option='plain', lamb = 0.0001, momentum = 0.1, max_epochs = 100)

{'delta': tensor([0.3898]),
 'beta': tensor([0.6750, 0.4713, 0.2900]),
 'gamma': tensor([-0.3281,  1.0489,  0.5946, -1.0341, -0.3051]),
 'converged': False,
 'iterations': 100,
 'final_delta_norm': 0.44334010283152264,
 'epochs_run': 100}

### fit with EMA

In [15]:
model_minibatch = DRGUMiniBatch(df, covariates=["x1", "x2"], treatment="treatment", response="response")
model_minibatch.fit(pairs_per_anchor=9, pairs_per_batch = 5000, option='plain', lamb = 0.0001, fisher_ema = 0.3)

{'delta': tensor([0.4949]),
 'beta': tensor([0.4721, 0.2749, 0.2054]),
 'gamma': tensor([ 0.0434,  0.4417,  1.2282, -0.4770, -0.1955]),
 'converged': False,
 'iterations': 10,
 'final_delta_norm': 0.4624425967534383,
 'epochs_run': 10}

## small batch

In [16]:
model_minibatch = DRGUMiniBatch(df, covariates=["x1", "x2"], treatment="treatment", response="response")
model_minibatch.fit(pairs_per_anchor=9, pairs_per_batch = 500, max_epochs = 100, option='plain', lamb = 0.0001, fisher_ema = 0.5, max_step_norm = 50)

{'delta': tensor([0.4220]),
 'beta': tensor([0.6688, 0.4706, 0.2316]),
 'gamma': tensor([-0.4433,  1.2216,  0.5973, -1.4065, -0.4536]),
 'converged': False,
 'iterations': 100,
 'final_delta_norm': 0.47534871101379395,
 'epochs_run': 100}

In [17]:
model_minibatch.fit(pairs_per_anchor=9, pairs_per_batch = 500, option='plain', lamb = 0.0001, momentum = 0.1)

{'delta': tensor([0.4726]),
 'beta': tensor([0.4721, 0.2641, 0.2142]),
 'gamma': tensor([ 0.0435,  0.3634,  0.9810, -0.3500, -0.0840]),
 'converged': False,
 'iterations': 10,
 'final_delta_norm': 0.3866267204284668,
 'epochs_run': 10}